# Image/frame baseline — EfficientNet-B0

PPT-aligned starter experiment. Requires a preprocessed face-crop dataset and a CSV manifest with `path,label,group_id,split,dataset`. This notebook intentionally does not fetch datasets or bypass access controls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/VeriSight_AI_Storage')
MANIFEST=BASE/'manifests/video_frames.csv'
assert MANIFEST.exists(), 'Create the approved dataset manifest first.'

In [ ]:
!pip -q install timm scikit-learn pandas pillow

In [ ]:
import pandas as pd
df=pd.read_csv(MANIFEST)
required={'path','label','group_id','split','dataset'}
assert required.issubset(df.columns)
assert set(df.label.unique()).issubset({'real','fake'})
assert set(df.split.unique()).issubset({'train','val','test'})
# Group leakage guard: one source video/identity must not cross splits.
assert (df.groupby('group_id').split.nunique() <= 1).all(), 'Leakage: group_id occurs in multiple splits.'
print(df.groupby(['dataset','split','label']).size())

In [ ]:
import torch, timm, numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
class FaceCropDataset(Dataset):
    def __init__(self, frame, root):
        self.frame=frame.reset_index(drop=True); self.root=Path(root)
        self.tf=transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]; p=Path(row.path); p=p if p.is_absolute() else self.root/p
        x=self.tf(Image.open(p).convert('RGB')); y=1 if row.label=='fake' else 0
        return x, torch.tensor(y,dtype=torch.long)
train_df=df[df.split=='train']; val_df=df[df.split=='val']
train_ds=FaceCropDataset(train_df, BASE); val_ds=FaceCropDataset(val_df, BASE)
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_ds,batch_size=16,shuffle=False,num_workers=2,pin_memory=True)

In [ ]:
model=timm.create_model('efficientnet_b0',pretrained=True,num_classes=2).to(DEVICE)
counts=train_df.label.value_counts(); weights=torch.tensor([len(train_df)/(2*counts.get('real',1)),len(train_df)/(2*counts.get('fake',1))],dtype=torch.float32,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=weights); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)
EPOCHS=3  # Increase only after a successful small smoke test.

In [ ]:
from sklearn.metrics import classification_report
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(); logits=model(x); loss=criterion(logits,y); loss.backward(); optimizer.step(); losses.append(loss.item())
    model.eval(); ys=[]; probs=[]
    with torch.no_grad():
        for x,y in val_loader:
            logits=model(x.to(DEVICE)); probs.extend(torch.softmax(logits,1)[:,1].cpu().numpy()); ys.extend(y.numpy())
    pred=(np.array(probs)>=.5).astype(int)
    print(epoch+1, 'train_loss', round(float(np.mean(losses)),4), 'val_acc', accuracy_score(ys,pred), 'val_f1', precision_recall_fscore_support(ys,pred,average='binary',zero_division=0)[2], 'val_auc', roc_auc_score(ys,probs) if len(set(ys))==2 else 'undefined')

In [ ]:
out=BASE/'models/image'; out.mkdir(parents=True,exist_ok=True)
torch.save({'state_dict':model.state_dict(),'architecture':'efficientnet_b0','class_to_index':{'real':0,'fake':1}},out/'efficientnet_b0_baseline.pt')
print('Saved:',out/'efficientnet_b0_baseline.pt')

## Before claiming results
Run a final untouched test split only after model selection. Add cross-dataset evaluation on Celeb-DF v2 and Deepfake-Eval-2024 under their terms. Report confusion matrix, accuracy, precision, recall, F1, ROC-AUC, speed, file size and limitations. Do not treat this small starter run as a final result.